# Build a RAG System — Day 3 Challenge

### What you are building

A question-answering system that reads **your documents** instead of making
things up.

Think of an **open-book exam**. The model is the student. Your documents are
the book. RAG is the bit that finds the right page and puts it in front of
the student before they answer.

**RAG** = **R**etrieval (find the right page) **A**ugmented (add it to the
question) **G**eneration (write the answer).

### How to run it

1. Run **cell 1**.
2. **Runtime → Restart session.**
3. **Runtime → Run all.**

Do not run cell 1 again after the restart.

### How to read it

Every step has three parts:

- a note **before** the code — what we are doing, and why
- the **code**
- a note **after** it — what the output means

**Read both notes.** The code is only about sixty lines. The thinking around
it is the lesson.

**About ten minutes in total**, mostly downloads.

## 1 · Install the libraries

**What we are doing:** installing the tools.

The version numbers are pinned on purpose. Without them you get whatever
was newest that day — and these libraries move classes between packages
quite often. When that happens, your code breaks four cells later with a
confusing error.

Pinned versions mean **your notebook works the same way tomorrow**.

**Takes 2–3 minutes. Then restart.**

In [ ]:
!pip install -q "langchain==1.4.6" "langchain-community==0.4.2" \
                "langchain-huggingface==1.2.3" "langchain-classic==1.0.8" \
                "langchain-text-splitters==1.1.3" \
                "transformers<5" sentence-transformers datasets faiss-cpu

# then: Runtime -> Restart session, once. You do not need to run this cell again.

### What you just saw

**The red `ERROR` text is normal. Ignore it.**

Look at which packages it names: `gradio`, `diffusers`, and Colab's
`requests`. **We never use any of them.** Installing our versions upset
some unrelated packages that came with Colab. Nothing is broken.

**Now restart: Runtime → Restart session, then Runtime → Run all.**

Why? Python cannot swap out a library that is already loaded. Without the
restart you might still be using the old versions.

## 2 · Load the documents

**What we are doing:** downloading real documents for our system to search.

### What is in this dataset

`dolly-15k` has 15,011 rows. Each row has four fields:

| field | what it is |
|---|---|
| `instruction` | the question |
| **`context`** | background text — **this is the "passage"** |
| `response` | the answer |
| `category` | the type of question |

**A row with a passage:**

- **question** — "When did Virgin Australia start operating?"
- **context** — *"Virgin Australia… commenced services on 31 August 2000…"*

Give someone that paragraph and they can answer the question. **That is
what a passage is: the text you need in order to answer.**

**A row with no passage:**

- **question** — "Why can camels survive without water?"
- **context** — empty

Nothing to attach. You answer that from your own head.

### Why it matters

**We only put the `context` column into our system.** The questions and
answers are thrown away.

So an empty `context` gives us nothing. That is why we delete those rows.

**Your system only knows what is in the `context` column.**

### Three things this cell does

1. **Load** the rows.
2. **Unmangle** — fix a bug in the loader that wraps every piece of text in
   quotes. `json.loads` undoes it.
3. **Filter and cap** — delete the empty rows, then keep only 500. All of
   them would take half an hour. 500 takes two minutes and teaches the same
   thing.

**Under a minute.**

In [ ]:
import json
from langchain_community.document_loaders import HuggingFaceDatasetLoader

dataset_name        = "databricks/databricks-dolly-15k"
page_content_column = "context"

loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
raw = loader.load()
print("rows loaded      :", len(raw))

# 1. undo the loader's json.dumps - NOT optional, see the warning below
def unmangle(text):
    try:
        return json.loads(text)
    except Exception:
        return text

for d in raw:
    d.page_content = unmangle(d.page_content)

# 2. drop the empty ones - this only works AFTER step 1
usable = [d for d in raw if d.page_content.strip()]
print("with real content:", len(usable))

data = usable[:500]                                      # 3. cap it
print("using            :", len(data))
print()
print(repr(data[0].page_content[:200]))   # repr, so you can SEE the escaping is gone

### What you just saw

```
rows loaded      : 15011
with real content: 4467
using            : 500
```

**15,011** rows in total.

**4,467** of them have a passage. The other **10,544 are empty** — those are
the "answer from your own head" questions.

**500** is our cap, to keep things fast.

**Look at the last line.** It starts at `'Virgin Australia` — straight at
the word.

Before the unmangle step it looked like `'"Virgin Australia`, with a quote
stuck on the front. That is the loader bug. It also made empty passages
come out as `""` — two characters, so **not** empty, so they slipped past
our filter.

## 2b · See it for yourself *(optional, 2 seconds)*

**What we are doing:** checking the above instead of trusting it.

The loader kept the other fields in `.metadata`, so we can count which
**categories** have a passage and which do not.

In [ ]:
from collections import Counter

print("categories WITH a passage (these became your index):")
for cat, n in Counter(d.metadata["category"] for d in usable).most_common():
    print(f"   {cat:22} {n:6,}")

print("\ncategories with NO passage (these were dropped):")
for cat, n in Counter(d.metadata["category"] for d in raw
                      if not d.page_content.strip()).most_common():
    print(f"   {cat:22} {n:6,}")

print("\n--- one row in full, all four fields ---")
d = usable[0]
print("category   :", d.metadata["category"])
print("instruction:", d.metadata["instruction"])
print("context    :", d.page_content[:160], "...")
print("response   :", str(d.metadata["response"])[:160], "...")

### What you just saw

**Two lists.**

The **top** list is the categories that come with text — questions about a
document someone gave you. These are your 4,467.

The **bottom** list is the categories with nothing attached — general
knowledge questions. These are the 10,544 we dropped.

**Those dropped rows are not broken.** They are fine for other purposes.
They are just useless to us, because we are building something that
searches text, and they have no text.

At the end, one full row is printed so you can see all four fields
together. **Only the `context` line goes into our system.**

## 3 · Cut the documents into chunks

**What we are doing:** cutting each document into small pieces.

**Why?** Think about looking something up in a book. You do not photocopy
the whole book — you photocopy **the page with the answer on it**.

Same idea. If we search whole documents, the one sentence we need gets
buried in four pages of other stuff, and the model has to find it. Often it
does not.

**So we cut everything into paragraph-sized pieces, and search those.**

Two settings:

- **`chunk_size=1000`** — at most 1000 characters per piece.
- **`chunk_overlap=150`** — each piece repeats the last 150 characters of
  the one before. So if a sentence gets cut in half, it still appears whole
  in one of the two pieces.

**Instant.**

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
docs = text_splitter.split_documents(data)

print(len(data), "documents ->", len(docs), "chunks")
print(docs[0].page_content[:200])

### What you just saw

```
500 documents -> N chunks
```

**More chunks than documents**, because long documents got cut up. Short
ones stayed whole.

Below that, the first chunk is printed. **Look at where it starts and where
it stops.** Does it begin mid-sentence? Does it cut a fact in half?

A chunk that stops halfway through a sentence can be found perfectly and
still give a useless answer. That is what the overlap is protecting
against.

**One thing to know:** 1000 characters is about 250 *tokens*, and the model
in the next cell stops reading at **256 tokens**. So we are right on the
edge — sometimes a little text at the end of a chunk is ignored, silently.

We kept it anyway. **Knowing you made that trade is worth a sentence in
your submission.**

## 4 · Turn text into numbers

**What we are doing:** loading the model that converts text into numbers.

**Why numbers?** A computer cannot compare two *meanings*. But it can
compare two *numbers* very easily.

So we use a model that reads a piece of text and gives back a list of
numbers. **Text that means similar things gets similar numbers.**

That is the trick behind everything here. It is why, in the next cell,
searching for *"cheesemaking"* will find a passage about *curds* — even
though they share no words at all.

**This cell does not do the chunks yet.** It just loads the model and tests
one sentence, so you find out here if the download failed — not two minutes
into the slow cell.

**About a minute the first time.**

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name    = "sentence-transformers/all-MiniLM-l6-v2",
    model_kwargs  = {"device": "cpu"},
    encode_kwargs = {"normalize_embeddings": True},   # official text says False - see below
)

v = embeddings.embed_query("This is a test document.")
print("dimensions:", len(v))
print("first three:", [round(x, 4) for x in v[:3]])

### What you just saw

```
dimensions: 384
first three: [...]
```

**Every piece of text becomes a list of 384 numbers.** That is what an
"embedding" is.

The three numbers shown are just the first three of the 384. **They mean
nothing on their own.** There is no "cheese number". Only the *distance
between two whole lists* means anything.

**`normalize_embeddings=True`** makes every list the same length. That makes
comparing two of them quick and simple.

The official challenge sets this to `False`, which is the opposite of what
you learned on Day 2. Either is fine — **but be able to say which you chose
and why.**

## 5 · Build the search index

**What we are doing:** converting every chunk into numbers and putting them
in FAISS.

**FAISS is a search engine for numbers.** Give it a list of numbers and it
finds the closest matches out of thousands, very fast.

**This is the slow cell.** Every chunk goes through the model one at a time.
One to two minutes. This is what the 500 cap was protecting you from —
without it you would wait twenty-five.

**Then we do something the official challenge skips.** Before adding any
language model, we run one search and print the result.

**Why?** A RAG system has two halves, and they break separately:

- the search finds the **wrong text**, or
- the search is fine and the model **misreads it**

Both give you a bad answer that looks exactly the same. **Test them one at
a time.**

**One to two minutes.**

In [ ]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(docs, embeddings)
print("indexed:", db.index.ntotal, "vectors")

# sanity check: does retrieval work at all, before you attach a model?
for d in db.similarity_search("What is cheesemaking?", k=2):
    print("-", d.page_content[:120].replace("\n", " "), "...")

### What you just saw

```
indexed: N vectors
- Cheesemaking (or caseiculture) is the craft of making cheese ...
```

**The vector count should match your chunk count from cell 3.** If it does
not, something went missing — find out what before continuing.

The lines underneath are the important bit. **That is a real search, and
there is no language model yet.** You asked a question and got back the two
closest chunks out of hundreds, using only numbers.

**Do this check every time.** If the chunks coming back are rubbish, the
problem is here — and changing the model in the next cell will not fix it.
Ten seconds now saves an hour of blaming the wrong half.

## 6 · The model that writes the answer

**What we are doing:** loading a small language model, and testing it alone.

**This is the step the official challenge gets wrong.**

There are two kinds of model, and they do very different jobs:

- **Extractive** — like a **highlighter pen**. It points at words that are
  already in the text. It cannot write anything new.
- **Generative** — like a **person**. It reads and writes an answer in its
  own words.

The official challenge loads an **extractive** one. But RAG is
Retrieval-Augmented **Generation** — the G means the model has to *write*.
A highlighter cannot do that.

**If you use the official one:** every cell goes green, you wait through the
slow embedding, and then the *last* cell fails with a confusing error.

We use `flan-t5-base`, which writes.

**About a minute for a 1 GB download.**

In [ ]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

gen = pipeline(
    "text2text-generation",          # a task LangChain accepts - see the warning
    model="google/flan-t5-base",
    max_new_tokens=128,
)
llm = HuggingFacePipeline(pipeline=gen)

print(llm.invoke("Answer in one sentence: what is cheese made from?"))

### What you just saw

```
Cheese is made from milk.
```

**Short, and that is normal.** `flan-t5-base` is a small model. It answers
in fragments, not paragraphs. Not a bug, and not what you are marked on.

**The point is that it wrote a sentence.** The official model cannot do that
at all.

**Now notice something important: we gave it no documents.** It answered
from what it learned during training.

That is exactly the behaviour the rest of this notebook is built to
control — and in cell 10 you will watch it do the same thing at a moment
when it really should not.

## 7 · Join the two halves

**What we are doing:** connecting the search to the model.

Right now you have two separate things: something that finds text, and
something that writes answers. This joins them.

**From now on, every question you ask will automatically:**

1. Turn your question into numbers
2. Search the index for the 4 closest chunks
3. Paste those 4 chunks into a prompt, with your question
4. Send it to the model
5. Hand back the answer — **and the chunks it used**

**Those five steps are RAG.** Everything before this was getting ready.

**Nothing will run** when you execute this cell. It only connects things.

**Instant.**

In [ ]:
from langchain_classic.chains import RetrievalQA

retriever = db.as_retriever(search_kwargs={"k": 4})

qa = RetrievalQA.from_chain_type(
    llm       = llm,
    chain_type= "stuff",                 # official says "refine" - see below
    retriever = retriever,
    return_source_documents=True,        # official says False - you want True
)
print("chain ready")

### What you just saw

```
chain ready
```

**Nothing ran.** This cell only connected the pieces. Three settings decided
how:

- **`k=4`** — four chunks per question. Think of it as a budget. More chunks
  means more for the model to read, and the useful one can get buried.
- **`chain_type="stuff"`** — put all four chunks in one prompt, ask once.
  The official text uses `"refine"`, which asks **once per chunk** and
  rewrites its own answer each time. Four times slower.
- **`return_source_documents=True`** — **the important one.** It makes the
  result include the chunks it used. That is what cell 9 needs. The official
  text sets this to `False`, which leaves you blind.

There is also a prompt hidden inside `"stuff"` that you never wrote, and it
gets sent with every question. It goes out in your name, so take a look:

```python
print(qa.combine_documents_chain.llm_chain.prompt.template)
```

## 8 · Ask it a question

**What we are doing:** running the whole thing for the first time.

One question goes in. Behind the scenes: it gets turned into numbers, four
chunks come back, they go into a prompt with your question, the model reads
it and writes a reply.

**Expect a short answer.** Small model. Normal.

**A few seconds.**

In [ ]:
question = "What is cheesemaking?"

result = qa.invoke({"query": question})      # official uses .run() - deprecated

print("Q:", question)
print("A:", result["result"])

### What you just saw

```
Q: What is cheesemaking?
A: ...
```

**If you got a sentence, the whole pipeline works.**

**But here is the problem.** Looking at this answer, you **cannot tell**
whether it came from your documents or from the model's own memory.

It looks exactly the same either way.

That is not a small detail. **It is the main problem with RAG systems** —
and it is why the next cell exists.

## 9 · Check where the answer came from

**What we are doing:** printing the chunks that were actually used.

**Why this matters more than anything else here.**

Two completely different problems give you the same bad answer:

- the right chunk was **never found** → a **search** problem
- the right chunk was found and the model **misread it** → a **model**
  problem

From the answer alone, you cannot tell which. **From the chunks, you can —
in ten seconds.** And they need completely different fixes.

We also count how many words of the answer appear in the chunks. Rough, but
useful.

**A few seconds.**

In [ ]:
result = qa.invoke({"query": question})

print("Q:", question)
print("A:", result["result"])
print("\nRetrieved", len(result["source_documents"]), "chunks:\n")
for i, d in enumerate(result["source_documents"], 1):
    print(f"[{i}] {d.page_content[:220].strip()}...\n")

# crude grounding check: how much of the answer appears in what was retrieved?
retrieved = " ".join(d.page_content.lower() for d in result["source_documents"])
words = [w for w in result["result"].lower().split() if len(w) > 3]
hits  = sum(w in retrieved for w in words)
print(f"grounded: {hits}/{len(words)} words appear in the retrieved text")

### What you just saw

**Three things.**

**The answer** — same as before.

**The four chunks that were used.** **Read chunk [1]. Does it actually say
what the answer says?** This is how you tell a search problem from a model
problem.

**`grounded: N/M words`** — how many words of the answer appear somewhere in
the chunks.

**It is a smoke alarm, not proof.** A high number means the answer probably
was not invented from nothing. It does **not** mean the answer is correct —
a reply can use all the right words and still say something the chunks never
said. Short words like "the" are skipped, because they appear everywhere and
would flatter the score.

**This is the paragraph to write in your submission.** Pick one question.
Say which chunk the answer came from. Say whether that chunk really supports
it.

Everything else here is library syntax that will change within a year. This
habit will not.

## 10 · Break it on purpose

**What we are doing:** asking questions our documents cannot answer.

**The real question about any system is not "does it work?" — it is "how
does it fail?"**

A system that answers well when it knows, and **makes things up confidently
when it does not**, is more dangerous than one that is simply bad. Because
nothing tells you which one you are getting.

**Three questions:**

1. One our documents **can** answer — the control.
2. One they probably cannot, but the model may remember from training.
3. One that is **impossible** — about an event after the dataset was made.

Watch questions 2 and 3. **Especially watch whether it ever says "I don't
know".**

**A few seconds.**

In [ ]:
for q in [
    "What is cheesemaking?",                      # should be in the corpus
    "What is the capital city of Uruguay?",       # almost certainly is not
    "Who won the 2026 World Cup?",                # cannot be - dolly predates it
]:
    r = qa.invoke({"query": q})
    print("Q:", q)
    print("A:", r["result"])
    print("   top chunk:", r["source_documents"][0].page_content[:90].strip(), "...\n")

### What you just saw

**Three questions, two very different kinds of failure.**

**"What is cheesemaking?"** — the control. Should work, and the chunk
underneath should clearly support it.

**"Capital of Uruguay?"** — the answer might be **right**. Now look at the
chunk underneath. **If that chunk does not say "Montevideo", the model
answered from its own memory and your whole search system was skipped.** It
got away with it this time.

**A right answer for the wrong reason is more dangerous than a wrong one** —
because nothing warns you.

**"Who won the 2026 World Cup?"** — this one **cannot** be right. The dataset
is older than the event. Whatever it said, it invented.

**And notice what it never did: say "I don't know".** Small models almost
never do.

Try adding "only answer from the context" to the prompt and run these again.
**It will probably still answer.** That is not your work failing — **that is
the finding.** A prompt instruction is a request, not a rule.

Which leaves the obvious question: *how often does this happen, and how
would you know?* **That is tomorrow's lesson.**

## Before you submit

1. **Runtime → Restart session and run all.** Your grader runs this on a
   fresh machine. Anything that only worked in your session will disappear.
2. **Check every cell has output under it.**
3. **Share → General access → Anyone with the link.** A private link is the
   most common reason a finished challenge scores nothing.

### What actually gets the marks

**Every cell running is the starting point, not the achievement.**

The marks are in four things you **noticed** — and three of them are things
that went wrong:

- **Cell 6** — the official model cannot write, and what you used instead.
- **Cell 9** — which chunk your answer came from, and whether it really
  supports it.
- **Cell 10** — a question it answers confidently and wrongly. And the
  difference between *inventing* an answer and *skipping the search* to get
  a right one.
- **Cell 3 or 4** — one setting you chose on purpose and can defend.

**A notebook that runs perfectly and notices none of this is a worse
submission than one that notices all four.**